# Prosper Loan Default Prediction

This notebook analyzes the Prosper loan dataset with the objective of predicting whether a borrower will default on a loan. We follow a structured approach: data loading and cleaning, feature engineering, preprocessing, model building, hyperparameter tuning, and evaluation. To ensure clarity and reproducibility, we consolidate multiple variants of each algorithm into a single cell using grid search or cross‑validation. The data preparation steps remain unchanged from the original workflow; only the modeling section has been reorganized.

## 1. Setup and Data Loading

In [ ]:
# Data analysis and manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing and modeling
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, balanced_accuracy_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Optional: XGBoost (install via pip if not available)
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


### Load the dataset

Replace `'prosperLoanData.csv'` with the path to your dataset. The dataset is large and contains 113 937 loans described by 81 variables.

In [ ]:
# Load the Prosper loan dataset
# Note: Make sure the CSV file is available in your working directory.
df = pd.read_csv('prosperLoanData.csv')
df.head()

## 2. Data Cleaning and Preprocessing

The cleaning steps below follow the original notebook. We remove columns with a high proportion of missing values (> 20 %), drop identifiers and leakage variables, convert date columns to datetime, impute missing values, and engineer additional features.

In [ ]:
# Percentage of missing values per column
missing_ratio = df.isna().mean().sort_values(ascending=False)

# Display the most missing columns
missing_ratio

### Drop columns with > 20 % missing values

In [ ]:
# Identify columns with more than 20 % missing values
missing_procentage = 0.2
high_missing_cols = missing_ratio[missing_ratio > missing_procentage].index
print(f'Columns with > {missing_procentage*100:.0f}% missing values:')
print(list(high_missing_cols))

# Drop these columns
df = df.drop(columns=high_missing_cols)
print(f'Shape after dropping high‑missing columns: {df.shape}')


### Remove obvious identifiers and leakage variables

Identifiers (e.g. listing keys or member IDs) and variables that depend on future information (e.g. payments and losses) can cause data leakage. We remove them before modeling.

In [ ]:
# Drop identifier columns
id_columns = ['ListingKey', 'ListingNumber', 'LoanKey', 'LoanNumber', 'MemberKey']
df = df.drop(columns=id_columns)

# Drop variables likely to leak information from the future
leakage_vars = [
    'LP_CustomerPayments', 'LP_CustomerPrincipalPayments', 'LP_InterestandFees',
    'LP_ServiceFees', 'LP_CollectionFees', 'LP_GrossPrincipalLoss',
    'LP_NetPrincipalLoss', 'LP_NonPrincipalRecoverypayments',
    'LoanCurrentDaysDelinquent', 'LoanStatus'
]
df = df.drop(columns=leakage_vars)

print(f'Shape after dropping leakage features: {df.shape}')

### Convert date columns

Many date columns are stored as strings. We convert them to datetime objects to extract meaningful features later. We also clean the `DateCreditPulled` column, which contains mixed formats.

In [ ]:
# List of date columns to convert
list_dates = ['ListingCreationDate', 'LoanOriginationDate', 'FirstRecordedCreditLine', 'ClosedDate']

# Display original dtypes
for col in list_dates:
    print(f'{col} type: {df[col].dtype}')

# Convert to datetime
for col in list_dates:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Clean the DateCreditPulled column (remove micro/nanoseconds)
df['DateCreditPulled'] = (
    df['DateCreditPulled']
        .astype(str)
        .str.replace(r'\.\d{6,9}$', '', regex=True)
    )
df['DateCreditPulled'] = pd.to_datetime(df['DateCreditPulled'], errors='coerce')

# Convert LoanOriginationQuarter from 'YYYY Qx' to a timestamp (quarter start)
tmp = df['LoanOriginationQuarter'].str.split(' ', expand=True)
df['LoanOriginationQuarter'] = (tmp[1] + 'Q' + tmp[0].str[1]).astype('period[Q]').dt.to_timestamp()

### Missing value exploration and imputation

We explore missing values by calculating the percentage of missing entries for each column. Depending on the variable type, we perform median imputation for numeric columns, mode imputation for low‑cardinality categorical variables, assign `'Unknown'` for high‑cardinality categorical variables, and impute date features via their year. A flag is added for missing `DebtToIncomeRatio`.

In [ ]:
# Identify columns with missing values
cols_with_na = df.columns[df.isna().sum() > 0]

# Count NAs per LoanStatusBinaire (intermediate check)
# (The details are omitted for brevity; see the original notebook for full exploration.)

# Define variable groups for imputation
num_vars = [
        'DebtToIncomeRatio', 'EmploymentStatusDuration', 'TotalCreditLinespast7years',
        'InquiriesLast6Months', 'CurrentDelinquencies', 'AmountDelinquent',
        'DelinquenciesLast7Years', 'PublicRecordsLast10Years'
    ]
cat_vars = ['BorrowerState']
high_card_vars = ['Occupation']
date_vars = ['ListingCreationDate', 'FirstRecordedCreditLine']

# Copy data for imputation
df_imputed = df.copy()

# Numeric variables: median imputation (with flag for DebtToIncomeRatio)
for col in num_vars:
    # Add a missing flag for DebtToIncomeRatio
    if col == 'DebtToIncomeRatio':
        df_imputed[col + '_missing'] = df[col].isna().astype(int)
    median_val = df[col].median()
    df_imputed[col] = df[col].fillna(median_val)

# Low‑cardinality categorical variables: fill with mode
for col in cat_vars:
    mode_val = df[col].mode()[0]
    df_imputed[col] = df[col].fillna(mode_val)

# High‑cardinality categorical variables: fill with 'Unknown'
for col in high_card_vars:
    df_imputed[col] = df[col].fillna('Unknown')

# Dates: extract year and impute year
for col in date_vars:
    df[col + '_year'] = df[col].dt.year
    df_imputed[col + '_year'] = df_imputed[col].dt.year
    median_year = df_imputed[col + '_year'].median()
    df_imputed[col + '_year'] = df_imputed[col + '_year'].fillna(median_year)

# Drop the original date variables after extracting year
df_imputed = df_imputed.drop(columns=date_vars)

### Feature engineering

We create several engineered features to capture borrower behaviour and credit risk: 

- **DebtPaymentRatio**: the ratio of monthly loan payment to stated income.
- **BadHistoryFlag**: a boolean flag indicating a poor credit history based on delinquencies, public records and recent inquiries.
- **RevolvingUtilization** and **RevolvingStressIndex**: measures of credit utilisation and stress on revolving credit lines.

In [ ]:
# DebtPaymentRatio
df_imputed['DebtPaymentRatio'] = (
    df_imputed['MonthlyLoanPayment'] / df_imputed['StatedMonthlyIncome']
    ).replace([np.inf, -np.inf], np.nan)

# BadHistoryFlag
df_imputed['BadHistoryFlag'] = (
    (df_imputed['CurrentDelinquencies'] > 0) |
    (df_imputed['DelinquenciesLast7Years'] > 2) |
    (df_imputed['PublicRecordsLast10Years'] > 0) |
    (df_imputed['InquiriesLast6Months'] > 5)
    ).astype(int)

# RevolvingUtilization and RevolvingStressIndex
df_imputed['RevolvingUtilization'] = (
    df_imputed['RevolvingCreditBalance'] /
    (df_imputed['RevolvingCreditBalance'] + df_imputed['AvailableBankcardCredit'])
    ).replace([np.inf, -np.inf], np.nan)
df_imputed['RevolvingStressIndex'] = (
    0.5 * df_imputed['RevolvingUtilization'] +
    0.5 * df_imputed['BankcardUtilization']
    )

### Encoding categorical variables

We prepare the data for modeling by encoding categorical variables. Nominal variables are one‑hot encoded, ordinal variables are encoded according to a predefined order, and binary variables are passed through untouched. All remaining numeric variables are left as is.

In [ ]:
# Identify variable types
nominal_vars = ['BorrowerState', 'Occupation', 'EmploymentStatus', 'ListingCategory (numeric)']
ordinal_vars = ['IncomeRange']
binary_vars = ['IsBorrowerHomeowner', 'CurrentlyInGroup', 'IncomeVerifiable', 'LoanStatusBinaire']

# All other numeric variables (excluding binary)
num_vars = df_imputed.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_vars = [col for col in num_vars if col not in binary_vars]

# Ordinal encoding for IncomeRange
df_imputed['IncomeRange'] = df_imputed['IncomeRange'].replace('Not employed', '$0')
income_order = ['$0', '$1-24,999', '$25,000-49,999', '$50,000-74,999', '$75,000-99,999', '$100,000+']
ordinal_encoder = OrdinalEncoder(categories=[income_order])

# One‑hot encoder for nominal variables
onehot_encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)

# ColumnTransformer to apply encoders
preprocessor = ColumnTransformer(transformers=[
        ('nominal', onehot_encoder, nominal_vars),
        ('ordinal', ordinal_encoder, ordinal_vars),
        ('numeric', 'passthrough', num_vars),
        ('binary', 'passthrough', binary_vars)
    ])

# Apply the transformations
df_encoded = preprocessor.fit_transform(df_imputed)

# Construct final DataFrame
encoded_columns = preprocessor.named_transformers_['nominal'].get_feature_names_out(nominal_vars).tolist()
df_final = pd.DataFrame(df_encoded, columns=encoded_columns + ordinal_vars + num_vars + binary_vars)

df_final.head()

## 3. Train/Validation/Test Split

We separate the cleaned dataset into features `X` and target `y` (the binary loan status), replace any remaining missing values with zeros, and split the data into training (60 %), validation (20 %) and test (20 %) sets using stratified sampling.

In [ ]:
# Separate target and features
y = df_final['LoanStatusBinaire'].astype(int)
X = df_final.drop(columns=['LoanStatusBinaire'])

# Replace any remaining NaNs with 0
X = X.fillna(0)

# Ensure categorical/binary columns are integer for certain models
for col in X.columns:
    if X[col].dtype == 'object' or X[col].dtype == 'bool':
        X[col] = X[col].astype(int)

# Split into train+val and test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
    )

# Further split train and validation (20 % of total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
    )

print('Train shape:', X_train.shape)
print('Validation shape:', X_val.shape)
print('Test shape:', X_test.shape)

## 4. Evaluation Metrics

We define a helper function to compute a set of commonly used metrics (accuracy, balanced accuracy, recall, F1 score, ROC‑AUC and Gini). This function will be used across models to report performance.

In [ ]:
def get_metrics(y_true, y_pred, y_proba):
    metrics = {}
    metrics['Accuracy'] = accuracy_score(y_true, y_pred)
    metrics['Balanced Accuracy'] = balanced_accuracy_score(y_true, y_pred)
    metrics['Recall class 1'] = recall_score(y_true, y_pred, pos_label=1)
    metrics['Recall class 0'] = recall_score(y_true, y_pred, pos_label=0)
    metrics['F1-score class 1'] = f1_score(y_true, y_pred, pos_label=1)
    metrics['ROC-AUC'] = roc_auc_score(y_true, y_proba)
    metrics['Gini'] = 2 * metrics['ROC-AUC'] - 1
    return metrics


## 5. Modeling and Hyperparameter Tuning

To streamline the analysis, we consolidate multiple versions of each algorithm into a single cell. For each model family we perform a grid search (or cross‑validation) over a range of hyperparameters, select the best configuration, and evaluate the resulting model on the validation and test sets.

### 5.1 Logistic Regression (L1 vs L2)

We search over L1 (Lasso) and L2 (Ridge) penalties and two values of the regularisation strength \(C\). The solver is set appropriately for each penalty. A pipeline is used to standardise features before fitting.

In [ ]:
# Pipeline with scaler and logistic regression
log_reg_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=5000, class_weight='balanced'))
    ])

# Hyperparameter grid: penalty, solver and C
log_reg_param_grid = {
        'classifier__penalty': ['l1', 'l2'],
        'classifier__C': [0.1, 0.3],
        # The solver must be consistent with the penalty; 'saga' works for both L1 and L2, 'lbfgs' only for L2
        'classifier__solver': ['saga', 'lbfgs']
    }

# Stratified K-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid search with ROC-AUC as the scoring metric
log_reg_grid = GridSearchCV(log_reg_pipe, log_reg_param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, refit=True)

# Fit the grid search
# log_reg_grid.fit(X_train, y_train)

# Best parameters and model (uncomment above line to run)
# print('Best Logistic Regression parameters:', log_reg_grid.best_params_)
# best_log_reg = log_reg_grid.best_estimator_

# Evaluate on validation set
# y_val_proba = best_log_reg.predict_proba(X_val)[:, 1]
# y_val_pred  = best_log_reg.predict(X_val)
# val_metrics  = get_metrics(y_val, y_val_pred, y_val_proba)
# print('Validation metrics (Logistic Regression):')
# for k, v in val_metrics.items():
#     print(f'{k:20s}: {v:.3f}')

# Evaluate on test set
# y_test_proba = best_log_reg.predict_proba(X_test)[:, 1]
# y_test_pred  = best_log_reg.predict(X_test)
# test_metrics = get_metrics(y_test, y_test_pred, y_test_proba)
# print('
Test metrics (Logistic Regression):')
# for k, v in test_metrics.items():
#     print(f'{k:20s}: {v:.3f}')

**Interpretation.**

In the original analysis, two versions of the logistic regression were tested:

- **L1 penalty (Lasso)**: balanced accuracy ≈ 0.664; recall for class 1 (defaulters) ≈ 0.675; recall for class 0 (non‑defaulters) ≈ 0.659; ROC‑AUC ≈ 0.736.
- **L2 penalty (Ridge)**: balanced accuracy ≈ 0.663; recall for class 1 ≈ 0.689; recall for class 0 ≈ 0.653; ROC‑AUC ≈ 0.735 (Gini ≈ 0.469).

The L1 and L2 versions performed very similarly, with a slight edge to the L1 penalty for balanced accuracy and discrimination. Both models showed moderate ability to separate good and bad borrowers, with ROC‑AUC around 0.74.

### 5.2 Random Forest

We combine the previously tested Random Forest configurations into a single grid search. The parameter grid searches over different numbers of trees, minimum samples per leaf, and minimum samples to split a node. The class imbalance is handled via `class_weight='balanced'`.

In [ ]:
# Base RandomForestClassifier
rf_clf = RandomForestClassifier(class_weight='balanced', max_depth=10, max_features='sqrt', random_state=42, n_jobs=-1)

# Hyperparameter grid
rf_param_grid = {
        'n_estimators': [300, 500],
        'min_samples_split': [5, 10],
        'min_samples_leaf': [2, 5, 10]
    }

rf_grid = GridSearchCV(rf_clf, rf_param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, refit=True)

# Fit the grid search
# rf_grid.fit(X_train, y_train)

# Print best params
# print('Best Random Forest parameters:', rf_grid.best_params_)

# best_rf = rf_grid.best_estimator_
# Evaluate on validation
# y_val_proba = best_rf.predict_proba(X_val)[:, 1]
# y_val_pred  = (y_val_proba >= 0.4).astype(int)  # threshold tuning if needed
# val_metrics = get_metrics(y_val, y_val_pred, y_val_proba)
# print('Validation metrics (Random Forest):')
# for k, v in val_metrics.items():
#     print(f'{k:20s}: {v:.3f}')

**Interpretation.**

The Random Forest in the original notebook achieved very high recall for the positive class (≈ 0.91) but struggled on the negative class (recall ≈ 0.37–0.38). The overall accuracy was modest (≈ 0.52) and the balanced accuracy around 0.64, indicating that the model heavily favoured identifying risky loans at the expense of incorrectly flagging many safe loans. Gini coefficients were around 0.47–0.50, suggesting moderate discriminatory power.

### 5.3 XGBoost

Extreme Gradient Boosting (XGBoost) is a powerful ensemble method that typically outperforms simpler models on complex tasks. We consolidate the variants explored in the original analysis by searching over the number of boosting rounds, tree depths, learning rates and regularisation parameters. Note that XGBoost may need to be installed (e.g. `pip install xgboost`).

In [ ]:
if XGBOOST_AVAILABLE:
    # Base model
    xgb_clf = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        use_label_encoder=False,
        random_state=42,
        n_jobs=-1
    )

    # Hyperparameter grid
    xgb_param_grid = {
        'n_estimators': [200, 300, 500],
        'max_depth': [4, 6, 8],
        'learning_rate': [0.05, 0.1],
        'subsample': [0.8],
        'colsample_bytree': [0.8],
        'reg_alpha': [0.0, 0.5],
        'reg_lambda': [1.0, 2.0]
    }
    
    xgb_grid = GridSearchCV(xgb_clf, xgb_param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, refit=True)

    # Fit the grid search
    # xgb_grid.fit(X_train, y_train)

    # print('Best XGBoost parameters:', xgb_grid.best_params_)
    
    # best_xgb = xgb_grid.best_estimator_
    # y_val_proba = best_xgb.predict_proba(X_val)[:, 1]
    # y_val_pred  = (y_val_proba >= 0.3).astype(int)  # threshold tuning as in original
    # val_metrics = get_metrics(y_val, y_val_pred, y_val_proba)
    # print('Validation metrics (XGBoost):')
    # for k, v in val_metrics.items():
    #     print(f'{k:20s}: {v:.3f}')
else:
    print('XGBoost is not installed. Please install it to run this section.')

**Interpretation.**

The XGBoost model demonstrated superior performance compared to the baseline logistic regression and Random Forest. In the original notebook it achieved an accuracy around 0.69, balanced accuracy ≈ 0.685, recall for the positive class ≈ 0.669 and recall for the negative class ≈ 0.702. The ROC‑AUC was approximately 0.752–0.759 (Gini ≈ 0.505–0.517). These numbers indicate a more balanced and effective discrimination between good and bad borrowers.

### 5.4 Support Vector Machine (SVM)

We implement a pipeline with feature scaling followed by an SVM classifier. We search over values of the regularisation parameter \(C\) and the kernel width parameter \(\gamma\). The model supports probability estimates to enable ROC‑AUC and threshold tuning.

In [ ]:
# Pipeline with scaler and SVC
svm_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42))
    ])

# Hyperparameter grid
svm_param_grid = {
        'classifier__C': [0.1, 1.0, 10.0],
        'classifier__gamma': ['scale', 'auto']
    }

svm_grid = GridSearchCV(svm_pipe, svm_param_grid, cv=cv, scoring='roc_auc', n_jobs=-1, refit=True)

# Fit the grid search
# svm_grid.fit(X_train, y_train)

# print('Best SVM parameters:', svm_grid.best_params_)

# best_svm = svm_grid.best_estimator_
# y_val_proba = best_svm.predict_proba(X_val)[:, 1]
# y_val_pred  = best_svm.predict(X_val)
# val_metrics = get_metrics(y_val, y_val_pred, y_val_proba)
# print('Validation metrics (SVM):')
# for k, v in val_metrics.items():
#     print(f'{k:20s}: {v:.3f}')

**Interpretation.**

Support Vector Machines can capture complex, non‑linear patterns via kernel functions. Although the original notebook did not include final metrics for SVM, tuning \(C\) and \(\gamma\) via grid search allows us to find a reasonable trade‑off between margin maximisation and misclassification errors.

## 6. Conclusions

After cleaning and feature engineering, several algorithms were trained to predict loan default risk. A summary of the findings:

| Model | Balanced Accuracy | Recall (class 1) | Recall (class 0) | ROC‑AUC / Gini | Comments |
|------|------------------|------------------|------------------|---------------|----------|
| Logistic Regression (L1 & L2) | ~0.66 | ~0.67–0.69 | ~0.65–0.66 | ROC‑AUC ≈ 0.74 (Gini ≈ 0.47) | Simple, interpretable baseline; moderate discrimination |
| Random Forest | ~0.64 | ~0.91 | ~0.37 | ROC‑AUC ≈ 0.74 (Gini ≈ 0.47–0.50) | Very high recall for defaulters but poor on safe loans; modest overall accuracy |
| XGBoost | ~0.69 | ~0.67 | ~0.70 | ROC‑AUC ≈ 0.75 (Gini ≈ 0.51) | Best overall trade‑off between classes; higher discrimination |
| SVM | — | — | — | — | Requires tuning; not evaluated in original results |

The Extreme Gradient Boosting model offered the best balance of sensitivity and specificity, achieving the highest balanced accuracy and AUC among the tested algorithms. Logistic regression and Random Forest models provided useful baselines but were either less discriminative or biased towards one class. Incorporating a grid search within each algorithm simplified the workflow and ensures fair comparison across models. Future work could involve more extensive hyperparameter tuning, ensemble stacking, or cost‑sensitive evaluation aligned with the business objectives of credit risk management.